In [114]:
!python ReportLabs.py 

✅ Resume generated: Karthikeyan_Baskaran_Resume.pdf


In [16]:
#import statements and keys


import os
from pathlib import Path
from dotenv import load_dotenv
import os
import yaml
import logging
import re
from datetime import datetime
from groq import Groq
import ast
import yaml
import time

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# In a notebook, we use Path.cwd() (Current Working Directory)
# This assumes your notebook is running inside your project folder.
# .parent goes up one level to your parent directory.
env_path = Path.cwd().parent / "environment.env"

# Load the .env file
if load_dotenv(dotenv_path=env_path):
    print("Environment variables loaded successfully!")
else:
    print("Error: Could not find or load the .env file.")

GROQ_API_KEY = os.getenv('GROQ_API_KEY')


Environment variables loaded successfully!


In [17]:
def llm_response_qwen(prompt: str, GROQ_API_KEY: str) -> str:
    """Generic function to get a response from the Groq Qwen LLM."""
    try:
        client = Groq(api_key=GROQ_API_KEY)
        response = client.chat.completions.create(
            model="qwen/qwen3-32b",
            messages=[
                {
                    "role": "system",
                    "content": "You are a professional career coach and resume-writing assistant. Your task is to craft strong, impactful, and industry-relevant resume bullet points that make my profile stand out for data analytics, data engineering roles and other type of analyst roles"
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.1,
            max_completion_tokens=4096,
            top_p=0.1,
            reasoning_effort="default",
            stream=True,
            stop=None
        )
        content = ""
        for chunk in response:
            c = chunk.choices[0].delta.content if chunk.choices[0].delta.content else ""
            content += c
        # Remove <think>...</think> block and clean up surrounding whitespace
        content = re.sub(r'\s*<think>.*?</think>\s*', '', content, flags=re.DOTALL).strip()
             
        return content
    except Exception as e:
        logging.error(f"An error occurred while communicating with the Groq API: {e}")
        return ""

In [18]:
def yamlcheck(content):
    try:
        raw_response = content.strip()
    except:
        None

    # Remove markdown code fences if Qwen accidentally includes them
    if raw_response.startswith("```"):
        raw_response = (raw_response.replace("```python", "").replace("```", "").strip()
        )

    try:
        # Safely convert the string representation of a list into a real Python list
        content = ast.literal_eval(raw_response)
    except Exception as e:
        print(f"Error parsing list: {e}")
        # Fallback: just split by lines if it completely fails
        content = [line.strip() for line in raw_response.split("\n") if line.strip()]
        
    return content

In [19]:
jobdescription = input('Enter the job description')

In [ ]:
#Load the content files

path = '/Users/karthik/Documents/Github/Colab/Resume.yaml'
with open(path, 'r') as f:
    resume = yaml.safe_load(f)
a = list(resume['Professional Experience'].keys())

#Load Destination Files
path = '/Users/karthik/Documents/Github/Colab/output copy.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)



#Create loop to get tailored initial responses with checking all the experiences are available

if (len(a) ==len(test['work_experience'])):
    for i in range(len(a)):
        experience = resume['Professional Experience'][a[i]]
        prompt1 = f"""[Task] Rewrite exactly 7 resume bullet points to align with the Job Description.

        [Constraints]
        - Truthfulness: Use ONLY metrics, tools, and software explicitly in the Resume. Never add/change tools or numbers.
        - Format: Return ONLY a raw, valid Python list of 7 strings on ONE continuous line. No markdown blocks (```), no \n, no indents. Start with [ and end with ].
        - Content: Blend problem, solution, and impact seamlessly. No labels like "Problem:" or "Impact:".

        [Example]
        Resume: - Built SQL tracking systems for supplier monitoring, cutting reporting time by 50%.
        Job: Optimize vendor workflows and leverage data analytics.
        Output: ["Optimized vendor workflows by building SQL tracking systems to automate supplier monitoring, cutting reporting time by 50%."]

        [Data]
        Resume: {experience}
        Job: {jobdescription}

        [Guardrail] Ensure exactly 7 elements are in the list. Zero invented skills/tools."""
        tailoredresponse = llm_response_qwen(prompt1, GROQ_API_KEY)
        cleanedresponse = yamlcheck(tailoredresponse)

        test['work_experience'][i]['achievements'] = cleanedresponse
        time.sleep(30)


    # Now saving to YAML will be perfectly clean and structured
    with open("output.yaml", "w", encoding="utf-8") as f:
        yaml.dump(test, f, sort_keys=False, default_flow_style=False)

    print("inital tailored resume collected")
else:
    print("There is not same number of experiences in source and destination")

2026-05-25 20:31:06,284 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-25 20:31:33,039 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-05-25 20:31:33,040 - INFO - Retrying request to /openai/v1/chat/completions in 33.000000 seconds
2026-05-25 20:32:06,324 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-25 20:32:34,644 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-05-25 20:32:34,646 - INFO - Retrying request to /openai/v1/chat/completions in 4.000000 seconds
2026-05-25 20:32:38,775 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-25 20:33:08,837 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-05-25 20:33:08,839 - INFO - Retrying re

inital tailored resume collected


In [20]:
#Load the content files

path = '/Users/karthik/Documents/Github/Colab/Resume.yaml'
with open(path, 'r') as f:
    resume = yaml.safe_load(f)
a = list(resume['Professional Experience'].keys())

#Load Destination Files
path = '/Users/karthik/Documents/Github/Colab/output copy.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)



#Create loop to get tailored initial responses with checking all the experiences are available

if (len(a) ==len(test['work_experience'])):
    for i in range(len(a)):
        experience = resume['Professional Experience'][a[i]]
        prompt1 = f"""Your task is to rewrite the resume bullet points provided below to better align with the professional tone of the provided Job Description.

        STRICT TRUTHFULNESS DIRECTIVE:
        You must ONLY use the exact metrics, tools, and software present in the input Resume. You are strictly forbidden from adding new technologies, tools, or numbers (e.g., if the resume says 'Sage ERP', do NOT change it to 'SAP' or add 'Python'). 

        OUTPUT FORMAT:
        - Combine the problem, solution, and impact into a single, seamless professional resume bullet point. Do NOT use explicit labels like "Problem:", "Solution:", or "Impact:".
        - Return ONLY a raw, valid Python list on a single line. No line breaks (\n), no indentation, and no markdown code blocks. The output must start with [ and end with ] on one continuous line.

        ### EXAMPLE OF CORRECT BEHAVIOR (FEW-SHOT) ###
        Input Resume:
        - Built custom SQL-based tracking systems that automated supplier performance monitoring, cutting manual reporting time by 50% and providing early warnings for dependency risks.
        Input Job:
        Looking for a Procurement Specialist to optimize vendor management workflows, streamline lead times, and leverage data analytics for stakeholder reporting.
        Correct Output:
        ["Optimized vendor management by building custom SQL-based tracking systems to automate supplier performance monitoring, cutting manual reporting time by 50% and mitigating supply chain dependency risks."]

        ### ACTUAL TASK ###
        Resume: {experience}
        Job: {jobdescription}

        FINAL GUARDRAIL: Do not invent any skills or tools. Look at the output array you just generated. If it contains any tool or number not found in the Resume section above, delete it immediately. Select and rewrite only the top 7 most relevant points that best align with the Job Description. Ensure exactly 7 elements are in the final Python list."""
        tailoredresponse = llm_response_qwen(prompt1, GROQ_API_KEY)
        cleanedresponse = yamlcheck(tailoredresponse)

        test['work_experience'][i]['achievements'] = cleanedresponse
        time.sleep(10)


    # Now saving to YAML will be perfectly clean and structured
    with open("output.yaml", "w", encoding="utf-8") as f:
        yaml.dump(test, f, sort_keys=False, default_flow_style=False)

    print("inital tailored resume collected")
else:
    print("There is not same number of experiences in source and destination")

2026-05-25 19:46:33,986 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-25 19:46:46,080 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-05-25 19:46:46,081 - INFO - Retrying request to /openai/v1/chat/completions in 50.000000 seconds
2026-05-25 19:47:36,225 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-25 19:47:47,878 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-05-25 19:47:47,880 - INFO - Retrying request to /openai/v1/chat/completions in 14.000000 seconds
2026-05-25 19:48:02,154 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-25 19:48:14,601 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-05-25 19:48:14,602 - INFO - Retrying r

inital tailored resume collected


In [21]:
prompt1

'Your task is to rewrite the resume bullet points provided below to better align with the professional tone of the provided Job Description.\n\n        STRICT TRUTHFULNESS DIRECTIVE:\n        You must ONLY use the exact metrics, tools, and software present in the input Resume. You are strictly forbidden from adding new technologies, tools, or numbers (e.g., if the resume says \'Sage ERP\', do NOT change it to \'SAP\' or add \'Python\'). \n\n        OUTPUT FORMAT:\n        - Combine the problem, solution, and impact into a single, seamless professional resume bullet point. Do NOT use explicit labels like "Problem:", "Solution:", or "Impact:".\n        - Return ONLY a raw, valid Python list on a single line. No line breaks (\n), no indentation, and no markdown code blocks. The output must start with [ and end with ] on one continuous line.\n\n        ### EXAMPLE OF CORRECT BEHAVIOR (FEW-SHOT) ###\n        Input Resume:\n        - Built custom SQL-based tracking systems that automated sup

In [22]:
cleanedresponse

['Developed interactive dashboards to monitor procurement KPIs and supplier performance, reducing reporting time in purchasing by 40% through automated data visualization and SQL-based analytics.',
 'Led data-driven RFx processes on digital platforms, achieving 8–12% procurement cost reductions by leveraging competitive analysis and strategic sourcing techniques.',
 'Executed supplier auctions using digital platforms, driving 8–12% cost savings through competitive bidding and advanced Excel (macros, pivot tables) for data analysis.',
 'Analyzed Linear Pricing Patterns (LPP) across key materials to identify cost anomalies, supporting standardized pricing frameworks and 15% cost savings in procurement.',
 'Built a robust supplier base by implementing double sourcing and localization strategies, improving supply chain efficiency by 20% through data-informed decision-making.',
 'Conducted over 5 supplier auctions, achieving 15% average cost savings by fostering competitive bidding and inte

In [25]:
test['work_experience']

[{'title': 'Procurement Buyer',
  'company': 'Decibel Cannabis Company',
  'dates': 'Dec 2025 - Present',
  'achievements': ['Automated supplier performance monitoring using SQL-based tracking systems to reduce manual reporting time by 50% and mitigate dependency risks through early warning alerts',
   'Engineered Power BI dashboards to transform messy procurement data into actionable insights, accelerating sourcing workflows and improving decision-making',
   'Ensured 99%+ data accuracy in Sage ERP by maintaining a single source of truth for supply chain visibility, supporting leadership with reliable end-to-end analytics',
   'Streamlined data mining processes via complex SQL queries to extract and transform large datasets into monthly performance reports, enhancing data accessibility',
   'Re-engineered purchase planning workflows for global suppliers with long lead times, achieving 98% production uptime and reducing stockout risks',
   'Architected a Star Schema data model by struc

In [69]:
path = '/Users/karthik/Documents/Github/Colab/output copy.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)

In [84]:
print(i)

0


In [86]:
test['work_experience'][1]['achievements']

['Reduced procurement costs by 15% by implementing data-driven supplier evaluation models, conducting cost analysis, and negotiating strategic vendor agreements across global supply chains.',
 'Improved purchasing efficiency by 20% through predictive demand forecasting and inventory planning models, strengthening supplier coordination and reducing procurement delays.',
 'Designed supplier scorecards and procurement KPI dashboards using Power BI and Excel, enabling leadership to evaluate supplier reliability, pricing competitiveness, quality, and service levels.',
 'Mitigated supply chain risks by developing alternate sourcing strategies, evaluating supplier capabilities, and proactively addressing lead-time constraints, shortages, and supplier performance issues impacting production timelines.',
 'Negotiated supplier contracts and commercial agreements that delivered measurable cost savings while maintaining compliance with procurement and operational standards.',
 'Managed sourcing ac

In [ ]:
test['work_experience'][0]['achievements'] = resume['Professional Experience'][a[i]]

['Developed SQL-driven supplier performance tracking systems that automated procurement reporting, reduced manual analysis time by 50%, and enabled proactive mitigation of supplier dependency risks across North American operations.',
 'Implemented interactive Power BI dashboards integrating procurement, inventory, and supplier data, improving visibility into pricing, lead times, vendor performance, and operational purchasing metrics.',
 'Re-engineered procurement planning strategies for long lead-time global suppliers, achieving 98% production uptime while minimizing material shortages and supply disruptions.',
 'Led strategic sourcing and competitive quotation initiatives that reduced procurement spend, diversified supplier networks, and strengthened supply continuity across critical categories.',
 'Improved supplier accountability and delivery performance through structured vendor engagement and KPI tracking, increasing on-time delivery rates by 30%.',
 'Managed end-to-end purchasing

In [77]:
test['work_experience'][0]['achievements'] = resume['Professional Experience'][a[i]]

In [80]:
test['work_experience'][0]

{'title': 'Procurement Buyer',
 'company': 'Decibel Cannabis Company',
 'dates': 'Dec 2025 - Present',
 'achievements': ['Built custom SQL-based tracking systems that automated supplier performance monitoring, cutting manual reporting time by 50% and providing early warnings for dependency risks.',
  'Designed interactive Power BI dashboards that turned messy procurement data into clear, actionable insights, helping the team speed up sourcing workflows.',
  "Maintained 99%+ data accuracy within Sage ERP, ensuring the leadership team had a 'single source of truth' for end-to-end supply chain visibility.",
  'Streamlined data mining processes by engineering complex queries to pull from large datasets, transforming raw numbers into monthly performance reports.',
  'Reduced stockouts (98% production uptime) by re-engineering purchase planning for global suppliers with long lead times.',
  'Maintained lean inventory levels by synchronizing the full PO lifecycle with real-time manufacturing 

In [54]:
resume['Professional Experience']

{'Decibel Cannabis Company': ['Built custom SQL-based tracking systems that automated supplier performance monitoring, cutting manual reporting time by 50% and providing early warnings for dependency risks.',
  'Designed interactive Power BI dashboards that turned messy procurement data into clear, actionable insights, helping the team speed up sourcing workflows.',
  "Maintained 99%+ data accuracy within Sage ERP, ensuring the leadership team had a 'single source of truth' for end-to-end supply chain visibility.",
  'Streamlined data mining processes by engineering complex queries to pull from large datasets, transforming raw numbers into monthly performance reports.',
  'Reduced stockouts (98% production uptime) by re-engineering purchase planning for global suppliers with long lead times.',
  'Maintained lean inventory levels by synchronizing the full PO lifecycle with real-time manufacturing demands and monthly forecasts, reducing excess stock by 20%.',
  'Spearheaded strategic sou

In [ ]:
decibelcc = resume['Professional Experience']

KeyError: 0

In [ ]:
# decibel = llm_response_qwen(prompt1, GROQ_API_KEY)

2026-05-19 23:14:03,276 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [44]:
decibel

'["Developed and implemented procurement strategies that improved lead time performance by 10% through optimized scheduling and supplier coordination","Streamlined procurement cycles by automating document tracking and expediting supplier deliverables using Hexagon/Smart Plant Materials, reducing project close-out delays by 15%","Led cross-functional stakeholder meetings to align procurement timelines with project milestones, ensuring 98% of scheduled deliverables met deadlines","Designed and executed EOI/RFP processes for critical equipment procurements, securing cost savings of 8% through competitive bidding and supplier negotiations","Maintained comprehensive procurement records and updated expeditor system modules, improving data accuracy for project reporting and audit readiness","Resolved supplier compliance issues through pre-qualification audits, reducing bid evaluation rework by 20% and accelerating award recommendations","Managed simultaneous procurement projects across multi

In [48]:


# Assuming 'qwen_response' is the raw string you get back from the model
try:
    raw_response = decibel.strip()
except:
    None

# Remove markdown code fences if Qwen accidentally includes them
if raw_response.startswith("```"):
    raw_response = (raw_response.replace("```python", "").replace("```", "").strip()
    )

try:
    # Safely convert the string representation of a list into a real Python list
    decibel = ast.literal_eval(raw_response)
except Exception as e:
    print(f"Error parsing list: {e}")
    # Fallback: just split by lines if it completely fails
    decibel = [line.strip() for line in raw_response.split("\n") if line.strip()]

# Now saving to YAML will be perfectly clean and structured
with open("test.yaml", "w", encoding="utf-8") as f:
    yaml.dump({"Decibel":decibel}, f, sort_keys=False, default_flow_style=False)

In [47]:
{"Decibel":decibel}

{'Decibel': ['Developed and implemented procurement strategies that improved lead time performance by 10% through optimized scheduling and supplier coordination',
  'Streamlined procurement cycles by automating document tracking and expediting supplier deliverables using Hexagon/Smart Plant Materials, reducing project close-out delays by 15%',
  'Led cross-functional stakeholder meetings to align procurement timelines with project milestones, ensuring 98% of scheduled deliverables met deadlines',
  'Designed and executed EOI/RFP processes for critical equipment procurements, securing cost savings of 8% through competitive bidding and supplier negotiations',
  'Maintained comprehensive procurement records and updated expeditor system modules, improving data accuracy for project reporting and audit readiness',
  'Resolved supplier compliance issues through pre-qualification audits, reducing bid evaluation rework by 20% and accelerating award recommendations',
  'Managed simultaneous proc